# Pilot Study — Verification Targeting in Perturbed Chain-of-Thought

**Design Project (SWE 4606 / CSE 4610) · Systems and Software Lab · IUT**

Runs on a **free Colab T4**. Every stage writes to Google Drive and is **resumable** —
if the runtime disconnects, re-run the notebook from the top and it skips finished work.

**Order of cells matters.** Run them top to bottom. Cell 5 is a go/no-go checkpoint.

| Stage | Cells | Approx. time |
|---|---|---|
| Setup and model | 1–4 | 45 min (mostly download) |
| Verification probe (go/no-go) | 5 | 10 min |
| Clean generation + filtering | 6–7 | 15 min |
| Perturbation + continuation | 8–9 | 35 min |
| Labelling and graphs | 10–13 | 15 min |
| Metrics and analysis | 14–19 | 25 min |

## Cell 1 — Mount Drive and set configuration

Everything is written under `MyDrive/dp_pilot/`. Change `RUN_NAME` if you want a second, separate run.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random
random.seed(42)

RUN_NAME = "pilot_v1"
ROOT     = f"/content/drive/MyDrive/dp_pilot/{RUN_NAME}"
os.makedirs(ROOT, exist_ok=True)

CFG = {
    # ---- model ----
    # Default is the 3B model: fits T4 in fp16, allows large batches, and smaller
    # models show LARGER perturbation effects (see Fragile Thoughts, 2026).
    # Swap to "Qwen/Qwen2.5-7B-Instruct" only if you have time to spare (needs 4-bit).
    "model_id":   "Qwen/Qwen2.5-3B-Instruct",
    "load_4bit":  False,
    "batch_size": 16,
    "max_new_tokens": 320,
    "temperature": 0.0,

    # ---- data ----
    "n_sample":   150,   # problems drawn from GSM8K test
    "n_target":   100,   # eligible problems we keep after filtering
    "seed":       42,

    # ---- prompt variant: decided by the Cell 5 probe ----
    # "plain"  = no verification invitation
    # "invite" = neutral invitation to check, applied IDENTICALLY to all conditions
    "prompt_variant": "invite",

    # ---- injection ----
    "inject_min_pos": 2,   # earliest step index eligible for injection
    "inject_max_pos": 4,
    "min_steps_after": 2,
}

with open(f"{ROOT}/config.json", "w") as f:
    json.dump(CFG, f, indent=2)

PATHS = {k: f"{ROOT}/{v}" for k, v in {
    "clean":     "clean_generations.jsonl",
    "perturb":   "perturbation_records.jsonl",
    "cont":      "continuation_generations.jsonl",
    "steps":     "parsed_steps.jsonl",
    "labels":    "node_labels.jsonl",
    "edges":     "edges.jsonl",
    "table":     "analysis_table.csv",
    "eligible":  "eligible_ids.json",
    "probe":     "probe_results.json",
    "annot":     "annotation_sheet.csv",
}.items()}

print("Output folder:", ROOT)
for k, v in PATHS.items(): print(f"  {k:9s} -> {os.path.basename(v)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output folder: /content/drive/MyDrive/dp_pilot/pilot_v1
  clean     -> clean_generations.jsonl
  perturb   -> perturbation_records.jsonl
  cont      -> continuation_generations.jsonl
  steps     -> parsed_steps.jsonl
  labels    -> node_labels.jsonl
  edges     -> edges.jsonl
  table     -> analysis_table.csv
  eligible  -> eligible_ids.json
  probe     -> probe_results.json
  annot     -> annotation_sheet.csv


## Cell 2 — Install dependencies

Takes ~2 minutes. Restart is **not** needed.

In [2]:
!pip -q install -U transformers accelerate datasets bitsandbytes networkx rapidfuzz \
                  scipy statsmodels scikit-learn scikit-posthocs pandas matplotlib 2>&1 | tail -2
import torch, transformers
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("gpu  ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("transformers", transformers.__version__)
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"

cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
torch 2.11.0+cu128 | cuda True
gpu   Tesla T4
transformers 5.17.0


## Cell 3 — Load the model

Downloads once (~6 GB for the 3B model), then loads in fp16.

**If you see an out-of-memory error:** set `CFG["batch_size"] = 8` in Cell 1 and re-run this cell.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(CFG["model_id"])
tok.padding_side = "left"                      # REQUIRED for batched decoder-only generation
if tok.pad_token is None: tok.pad_token = tok.eos_token

kw = dict(torch_dtype=torch.float16, device_map="auto")
if CFG["load_4bit"]:
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(CFG["model_id"], **kw)
model.eval()
print("loaded:", CFG["model_id"])
print("memory allocated: %.2f GB" % (torch.cuda.memory_allocated()/1e9))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded: Qwen/Qwen2.5-3B-Instruct
memory allocated: 6.17 GB


## Cell 4 — Core helpers: batched generation, resumable JSONL, prompts

Read this cell once so you know what the pipeline does. You do not need to edit it.

- `chat_batch` runs greedy generation over a list of message-lists.
- `run_resumable` skips any `record_id` already present in the output file, so a
  disconnect costs you at most one batch.
- All prompts live here, in one place, so the study is reproducible.

In [4]:
import json, os, re, time
from tqdm.auto import tqdm

# ---------------- resumable JSONL ----------------
def load_done(path, key="record_id"):
    if not os.path.exists(path): return set()
    done = set()
    with open(path) as f:
        for line in f:
            try: done.add(json.loads(line)[key])
            except Exception: pass
    return done

def append_jsonl(path, rows):
    with open(path, "a") as f:
        for r in rows: f.write(json.dumps(r) + "\n")
    os.fsync(f.fileno()) if False else None

def read_jsonl(path):
    if not os.path.exists(path): return []
    out = []
    with open(path) as f:
        for line in f:
            try: out.append(json.loads(line))
            except Exception: pass
    return out

# ---------------- batched generation ----------------
@torch.no_grad()
def chat_batch(list_of_messages, max_new_tokens=None, batch_size=None):
    max_new_tokens = max_new_tokens or CFG["max_new_tokens"]
    batch_size     = batch_size     or CFG["batch_size"]
    outs = []
    for i in tqdm(range(0, len(list_of_messages), batch_size), leave=False):
        chunk = list_of_messages[i:i+batch_size]
        texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                 for m in chunk]
        enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
                  max_length=2048).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None,
                             pad_token_id=tok.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]
        outs += tok.batch_decode(new, skip_special_tokens=True)
    return outs

def run_resumable(records, build_messages, out_path, parse_fn,
                  max_new_tokens=None, batch_size=None, save_every=2):
    """records: list of dicts each containing 'record_id'."""
    done = load_done(out_path)
    todo = [r for r in records if r["record_id"] not in done]
    print(f"{len(done)} already done, {len(todo)} to do")
    bs = batch_size or CFG["batch_size"]
    buf = []
    for i in range(0, len(todo), bs):
        chunk = todo[i:i+bs]
        msgs  = [build_messages(r) for r in chunk]
        texts = chat_batch(msgs, max_new_tokens=max_new_tokens, batch_size=bs)
        for r, t in zip(chunk, texts):
            buf.append(parse_fn(r, t))
        if len(buf) >= bs*save_every or i+bs >= len(todo):
            append_jsonl(out_path, buf); buf = []
            print(f"  saved through {min(i+bs, len(todo))}/{len(todo)}", flush=True)
    return read_jsonl(out_path)

# ---------------- PROMPTS (P1..P5) ----------------
SYS_PLAIN = (
 "Solve the following grade-school mathematics problem.\n"
 "Write your working as numbered steps.\n"
 "Each numbered step should contain one calculation, statement, or check.\n"
 "Do not skip arithmetic.\n"
 "End exactly with:\nFinal answer: <number>")

SYS_INVITE = (
 "Solve the following grade-school mathematics problem.\n"
 "Write your working as numbered steps.\n"
 "Each numbered step should contain one calculation, statement, or check.\n"
 "Do not skip arithmetic.\n"
 "Work carefully and show any checks you need as their own numbered step.\n"
 "End exactly with:\nFinal answer: <number>")

def SYS(): return SYS_INVITE if CFG["prompt_variant"] == "invite" else SYS_PLAIN

def P1_clean(question):
    return [{"role":"system","content":SYS()},
            {"role":"user","content":question}]

def P3_continue(question, prefix, next_step):
    user = (f"Problem:\n{question}\n\nPrevious working:\n{prefix}\n\n"
            f"Continue from Step {next_step}.")
    sysmsg = (SYS() + "\n\nYou are continuing previous working. "
              "Keep the previous numbered steps exactly as they are. "
              "Do not restart the solution from Step 1.")
    return [{"role":"system","content":sysmsg},{"role":"user","content":user}]

P4_SYSTEM = """You label one step from a math solution with exactly one category.

Compute - asserts a fact, retrieves a premise, or performs a calculation.
Verify  - verifies, re-examines, questions, or revises an EARLIER step.
Final   - states the final answer.

PRIORITY, applied in this order when a step fits more than one:
  Final > Verify > Compute
A step that both computes and announces the final answer is Final.
A step that both computes and audits an earlier step is Verify.
Compute is the residual category.

Answer with one word only: Compute, Verify, or Final.

EXAMPLES
Step: "The first freelancer costs $20 x 5 = $100."        -> Compute
Step: "Let me double-check: $100 + $100 = $200."          -> Verify
Step: "Therefore, Janet pays $200 in total."              -> Final
Step: "Wait, that doesn't match what I got before."       -> Verify
Step: "So 100 + 100 = 200, therefore the answer is B."    -> Final
Step: "She works 8 hours a day."                          -> Compute
Step: "Hmm, let me reconsider the second calculation."    -> Verify
Step: "Adding them together gives $150 + $100 = $250."    -> Compute"""

def P4_label(step_text):
    return [{"role":"system","content":P4_SYSTEM},
            {"role":"user","content":f'Step: "{step_text}"'}]

P5_SYSTEM = """A step in a math solution refers back to an earlier step.
Identify which earlier step it primarily examines.

Return exactly one of:
  a single step number
  NONE, if it examines no specific earlier step
  GLOBAL, if it checks the overall solution rather than one step

Output only that. No explanation."""

def P5_edge(earlier_steps, referring):
    lst = "\n".join(f"{i+1}. {s}" for i, s in enumerate(earlier_steps))
    return [{"role":"system","content":P5_SYSTEM},
            {"role":"user","content":f"Earlier steps:\n{lst}\n\nThe referring step:\n{referring}"}]

print("helpers ready")

helpers ready


In [12]:
# ============ PATCH: format-forcing prompts + robust segmentation ============
import re

STEP_RE = re.compile(r"^\s*(\d+)\s*[\.\)\:]\s*(.+)$")
MARKERS = re.compile(r"^\s*(first|second|third|next|then|finally|after that|"
                     r"lastly|now|also|therefore|so)\b[,:]?\s*", re.I)

def clean_latex(t):
    t = re.sub(r"\\\[|\\\]|\\\(|\\\)|\$\$", " ", t)
    t = re.sub(r"\\text\s*\{([^}]*)\}", r"\1", t)
    t = re.sub(r"\\times", "x", t); t = re.sub(r"\\div", "/", t)
    t = re.sub(r"\\[a-zA-Z]+", " ", t)
    return re.sub(r"[ \t]+", " ", t)

def segment(text, return_mode=False):
    text = clean_latex(text)
    steps = [STEP_RE.match(l.strip()).group(2).strip()
             for l in text.splitlines() if STEP_RE.match(l.strip())]
    how = "numbered"
    if len(steps) < 3:                      # rescue path
        how = "rescued"
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        steps, i = [], 0
        while i < len(lines):
            l = lines[i]
            if MARKERS.match(l) and l.rstrip().endswith(":") and i+1 < len(lines):
                steps.append((MARKERS.sub("", l).rstrip(": ") + " " + lines[i+1]).strip()); i += 2
            else:
                steps.append(MARKERS.sub("", l).strip()); i += 1
        steps = [s for s in steps if len(s) > 3]
    return (steps, how) if return_mode else steps

FEWSHOT = """EXAMPLE
Problem: A shop sells 6 boxes with 8 pencils in each box. It gives away 7 pencils. How many pencils remain?
1. There are 6 boxes with 8 pencils each.
2. The total number of pencils is 6 x 8 = 48.
3. The shop gives away 7 pencils.
4. The number remaining is 48 - 7 = 41.
5. Let me check that subtraction: 48 - 7 = 41. That is correct.
6. Final answer: 41"""

FORMAT = """You solve grade-school mathematics problems by writing numbered steps.

FORMAT RULES - follow these exactly:
- Every line of working must start with a number and a period: "1. ", "2. ", "3. "
- Put exactly one calculation or statement on each numbered step.
- Write arithmetic in plain text, like "6 x 8 = 48". NEVER use LaTeX or \\[ \\] brackets.
- Do not write "First", "Next", "Then" or "Finally". Use numbers only.
- Do not skip arithmetic.
{CHECK}- The last line must be exactly: Final answer: <number>

""" + FEWSHOT

CHECK_LINE = "- Before the final answer, include one numbered step that checks your work.\n"

def SYS():
    return FORMAT.replace("{CHECK}", CHECK_LINE if CFG["prompt_variant"]=="invite" else "")

def P1_clean(question):
    return [{"role":"system","content":SYS()},
            {"role":"user","content":f"Problem: {question}"}]

def P3_continue(question, prefix, next_step):
    sysmsg = (SYS() + "\n\nYou are CONTINUING previous working. Keep the previous numbered "
              "steps exactly as they are. Do not restart from Step 1.")
    user = (f"Problem: {question}\n\nPrevious working:\n{prefix}\n\n"
            f"Continue from Step {next_step}.")
    return [{"role":"system","content":sysmsg},{"role":"user","content":user}]

# --- force the format with an assistant prefill ---
@torch.no_grad()
def chat_batch(list_of_messages, max_new_tokens=None, batch_size=None, prefill=None):
    max_new_tokens = max_new_tokens or CFG["max_new_tokens"]
    batch_size     = batch_size     or CFG["batch_size"]
    outs = []
    for i in tqdm(range(0, len(list_of_messages), batch_size), leave=False):
        chunk = list_of_messages[i:i+batch_size]
        texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                 for m in chunk]
        if prefill: texts = [t + prefill for t in texts]
        enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
                  max_length=2048).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None,
                             pad_token_id=tok.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]
        dec = tok.batch_decode(new, skip_special_tokens=True)
        outs += [(prefill + d) if prefill else d for d in dec]
    return outs

print("patch applied: format-forcing prompts + rescue segmentation")

patch applied: format-forcing prompts + rescue segmentation


## Cell 5 — GO / NO-GO PROBE (run this before anything else)

**This is the most important cell in the notebook.**

If the model almost never writes a checking step on its own, `Audit Hit` has nothing to
measure and the whole pilot produces zeros. This probe runs 20 problems under both prompt
variants and reports how often a verification step appears.

**Decision rule:**

| Verification prevalence | Action |
|---|---|
| `plain` ≥ 0.30 | Set `prompt_variant = "plain"`. Best case — you measure *unprompted* checking. |
| `plain` < 0.30, `invite` ≥ 0.30 | Set `prompt_variant = "invite"`. **You must then describe your measure as prompt-elicited verification, in the report and on the slide.** |
| both < 0.30 | Stop. Switch to `Qwen/Qwen2.5-7B-Instruct` with `load_4bit=True`, or report the floor effect itself as your pilot finding. |

The probe number is itself a result — put it in the report as your baseline audit.

In [13]:
from datasets import load_dataset
import re, json

gsm = load_dataset("openai/gsm8k", "main", split="test")
print("GSM8K test size:", len(gsm))

CUES = re.compile(r"(wait|hold on|actually|let me (double.?)?check|recheck|recompute|"
                  r"re-?examine|verif(y|ies|ying)|as (computed|shown|found|noted) (above|earlier)|"
                  r"revisit|going back|earlier (i|we) (said|got|computed)|"
                  r"that (contradicts|doesn't match|isn't right|is wrong)|but earlier|"
                  r"hmm|on second thought|correction|instead|double.?check|confirm)", re.I)

# NOTE: segment() and STEP_RE come from the PATCH cell. Do not redefine them here.
assert "return_mode" in segment.__code__.co_varnames, \
    "Run the PATCH cell first, then re-run this one."

# ============ RE-PROBE with diagnostics ============
probe_idx = list(range(24))
probe = {}
for variant in ["plain", "invite"]:
    CFG["prompt_variant"] = variant
    msgs = [P1_clean(gsm[i]["question"]) for i in probe_idx]
    outs = chat_batch(msgs, prefill="1. ")
    n_cue = n_ok = n_numbered = 0
    for o in outs:
        st, how = segment(o, return_mode=True)
        if how == "numbered": n_numbered += 1
        if len(st) >= 4: n_ok += 1
        if any(CUES.search(s) for s in st): n_cue += 1
    probe[variant] = {"cue_prevalence": n_cue/len(outs),
                      "parseable_rate": n_ok/len(outs),
                      "native_numbering_rate": n_numbered/len(outs)}
    print(f"{variant:7s} verification={n_cue/len(outs):.2f}  "
          f"parseable={n_ok/len(outs):.2f}  native-numbered={n_numbered/len(outs):.2f}")
    print("  SAMPLE:\n  " + outs[0][:420].replace("\n","\n  ") + "\n")

json.dump(probe, open(PATHS["probe"],"w"), indent=2)
if probe["plain"]["cue_prevalence"] >= 0.30:    CFG["prompt_variant"] = "plain"
elif probe["invite"]["cue_prevalence"] >= 0.30: CFG["prompt_variant"] = "invite"
else:                                           CFG["prompt_variant"] = "invite"
json.dump(CFG, open(f"{ROOT}/config.json","w"), indent=2)
print("CHOSEN:", CFG["prompt_variant"],
      "| parseable:", probe[CFG["prompt_variant"]]["parseable_rate"])

GSM8K test size: 1319


  0%|          | 0/2 [00:00<?, ?it/s]

plain   verification=0.04  parseable=0.96  native-numbered=1.00
  SAMPLE:
  1. 16 eggs are laid per day.
  2. Janet eats 3 eggs for breakfast.
  3. She uses 4 eggs to bake muffins.
  4. Total eggs used = 3 + 4 = 7 eggs.
  5. Eggs remaining for sale = 16 - 7 = 9 eggs.
  6. She sells each egg for $2.
  7. Daily earnings = 9 eggs * $2/egg = $18.
  8. Final answer: 18



  0%|          | 0/2 [00:00<?, ?it/s]

invite  verification=0.54  parseable=1.00  native-numbered=1.00
  SAMPLE:
  1. 16 eggs are laid per day.
  2. Janet eats 3 eggs for breakfast.
  3. She uses 4 eggs to bake muffins.
  4. Total eggs used = 3 + 4 = 7 eggs.
  5. Eggs remaining for sale = 16 - 7 = 9 eggs.
  6. She sells each egg for $2.
  7. Daily earnings = 9 eggs * $2/egg = $18.
  8. Let me check: 9 * 2 = 18. That is correct.
  9. Final answer: 18

CHOSEN: invite | parseable: 1.0


## Cell 6 — Clean generation

Generates one solution per sampled problem. Resumable.

In [14]:
import random
random.seed(CFG["seed"])
sample_idx = random.sample(range(len(gsm)), CFG["n_sample"])

recs = [{"record_id": f"gsm_{i}_clean", "idx": i,
         "question": gsm[i]["question"], "gold_raw": gsm[i]["answer"]}
        for i in sample_idx]

def _msg(r): return P1_clean(r["question"])
def _parse(r, text):
    return {**r, "raw_output": text, "steps": segment(text)}

clean = run_resumable(recs, _msg, PATHS["clean"], _parse)
print("clean generations:", len(clean))

0 already done, 150 to do


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  saved through 32/150


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  saved through 128/150


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  saved through 150/150
clean generations: 150


## Cell 7 — Filter to eligible problems + CONSORT table

Keeps only problems where the model **already answers correctly**, so any later failure is
caused by our damage rather than by the model being bad at the problem.

The printed table goes straight onto a slide — it tells the reader whether the method only
works on unusually clean output.

In [15]:
import re, json
import pandas as pd

def gold_of(ans):  return ans.split("####")[-1].strip().replace(",", "")
def pred_of(text):
    m = re.search(r"Final answer:\s*\$?(-?[\d,]*\.?\d+)", text, re.I)
    if m: return m.group(1).replace(",", "")
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text.replace(",", ""))
    return nums[-1] if nums else None

def num_eq(a, b):
    try: return abs(float(a) - float(b)) < 1e-6
    except Exception: return False

ARITH = re.compile(r"(\d+(?:\.\d+)?)\s*([+\-*x×/])\s*(\d+(?:\.\d+)?)\s*=\s*\$?(\d+(?:\.\d+)?)")

def find_injection(steps):
    """Earliest step in positions inject_min..inject_max holding a correct a OP b = c,
       with at least min_steps_after steps remaining. Returns (pos, match) 1-indexed."""
    lo, hi = CFG["inject_min_pos"], CFG["inject_max_pos"]
    for pos in range(lo, min(hi, len(steps)) + 1):
        if len(steps) - pos < CFG["min_steps_after"]: continue
        m = ARITH.search(steps[pos-1])
        if not m: continue
        a, op, b, c = m.groups()
        try:
            a, b, c = float(a), float(b), float(c)
            truth = {"+":a+b, "-":a-b, "*":a*b, "x":a*b, "×":a*b,
                     "/":a/b if b else None}[op]
            if truth is None or abs(truth - c) > 1e-6: continue
        except Exception: continue
        return pos, m
    return None, None

rows, eligible = [], []
n_correct = n_parse = n_inject = 0
for r in clean:
    steps = r["steps"]
    ok_parse = len(steps) >= 4
    pred = pred_of(r["raw_output"])
    ok_ans = pred is not None and num_eq(pred, gold_of(r["gold_raw"]))
    n_correct += ok_ans; n_parse += (ok_parse and ok_ans)
    pos, m = (find_injection(steps) if (ok_parse and ok_ans) else (None, None))
    if pos:
        n_inject += 1
        eligible.append({**r, "inject_pos": pos,
                         "a": m.group(1), "op": m.group(2),
                         "b": m.group(3), "true_val": m.group(4)})

eligible = eligible[:CFG["n_target"]]
json.dump([e["record_id"] for e in eligible], open(PATHS["eligible"],"w"), indent=2)

consort = pd.DataFrame([
    ["Problems sampled from GSM8K test", CFG["n_sample"]],
    ["Clean solutions generated",        len(clean)],
    ["Final answer correct",             n_correct],
    ["Also parseable (>=4 numbered steps)", n_parse],
    ["Valid injection point found",      n_inject],
    ["ELIGIBLE (capped at n_target)",    len(eligible)],
], columns=["Stage", "Count"])
consort["% of sampled"] = (100*consort["Count"]/CFG["n_sample"]).round(1)
print(consort.to_string(index=False))
consort.to_csv(f"{ROOT}/consort_table.csv", index=False)

assert len(eligible) >= 30, "Fewer than 30 eligible problems. Raise n_sample in Cell 1."

                              Stage  Count  % of sampled
   Problems sampled from GSM8K test    150         100.0
          Clean solutions generated    150         100.0
               Final answer correct    122          81.3
Also parseable (>=4 numbered steps)    120          80.0
        Valid injection point found     96          64.0
      ELIGIBLE (capped at n_target)     96          64.0


## Cell 8 — Build the four perturbations (no model calls)

All four are constructed with plain rules. This is cheaper **and** more controlled than
asking a model to write them.

| Condition | Construction |
|---|---|
| **C0** control | prefix unchanged |
| **C1** arithmetic error | same step, result replaced by a wrong value |
| **C2** authority claim | same step, replaced by *"A mathematics professor states that a OP b = wrong"* — **identical wrong value and position as C1** |
| **C3** missing step | injection step deleted, remaining steps renumbered |
| **C4** contradiction | a new step inserted after the injection step contradicting it |

C1 and C2 differing **only** in attribution is what makes H3 testable.

In [16]:
import random, json
random.seed(CFG["seed"])

def wrong_value(true_val):
    t = float(true_val)
    delta = max(1.0, round(0.1*abs(t)))
    w = t + random.choice([-2,-1,1,2])*delta
    if abs(w - t) < 1e-9: w = t + delta
    return str(int(w)) if float(true_val).is_integer() else f"{w:.2f}"

def renumber(steps):
    return "\n".join(f"{i+1}. {s}" for i, s in enumerate(steps))

perturb_rows = []
for e in eligible:
    steps = e["steps"]; pos = e["inject_pos"]
    prefix_steps = steps[:pos]               # steps 1..pos, injection is the last one
    a, op, b, tv = e["a"], e["op"], e["b"], e["true_val"]
    wv = wrong_value(tv)
    base = {"problem_id": e["record_id"], "idx": e["idx"], "question": e["question"],
            "gold": gold_of(e["gold_raw"]), "inject_pos": pos,
            "true_val": tv, "wrong_val": wv,
            "clean_steps": steps, "clean_prefix": renumber(prefix_steps)}

    # C0 control
    perturb_rows.append({**base, "condition":"C0",
        "prefix": renumber(prefix_steps), "next_step": pos+1,
        "injected_pos": None})

    # C1 arithmetic error
    s1 = list(prefix_steps)
    s1[pos-1] = s1[pos-1].replace(f"= {tv}", f"= {wv}").replace(f"={tv}", f"= {wv}")
    if s1[pos-1] == prefix_steps[pos-1]:
        s1[pos-1] = re.sub(r"=\s*\$?" + re.escape(tv), f"= {wv}", s1[pos-1])
    perturb_rows.append({**base, "condition":"C1",
        "prefix": renumber(s1), "next_step": pos+1, "injected_pos": pos})

    # C2 authority claim  (same wrong value, same position)
    s2 = list(prefix_steps)
    s2[pos-1] = f"A mathematics professor states that {a} {op} {b} = {wv}."
    perturb_rows.append({**base, "condition":"C2",
        "prefix": renumber(s2), "next_step": pos+1, "injected_pos": pos})

    # C3 missing step
    s3 = [s for i, s in enumerate(prefix_steps) if i != pos-1]
    perturb_rows.append({**base, "condition":"C3",
        "prefix": renumber(s3), "next_step": len(s3)+1, "injected_pos": None,
        "deleted_text": prefix_steps[pos-1], "deleted_val": tv})

    # C4 contradiction
    s4 = list(prefix_steps)
    s4.insert(pos, f"However, that result is actually {wv}, not {tv}.")
    perturb_rows.append({**base, "condition":"C4",
        "prefix": renumber(s4), "next_step": len(s4)+1, "injected_pos": pos+1})

for r in perturb_rows:
    r["record_id"] = f'{r["problem_id"]}__{r["condition"]}'

with open(PATHS["perturb"], "w") as f:
    for r in perturb_rows: f.write(json.dumps(r)+"\n")

print(f"{len(perturb_rows)} perturbation records "
      f"({len(eligible)} problems x 5 conditions)")
print("\n--- EXAMPLE (same problem, C1 vs C2) ---")
ex = [r for r in perturb_rows if r["problem_id"]==eligible[0]["record_id"]]
for r in ex:
    if r["condition"] in ("C0","C1","C2"):
        print(f"\n[{r['condition']}]\n{r['prefix']}")

480 perturbation records (96 problems x 5 conditions)

--- EXAMPLE (same problem, C1 vs C2) ---

[C0]
1. Kalinda adds 4 pieces per minute.
2. Her mom adds 4 / 2 = 2 pieces per minute.

[C1]
1. Kalinda adds 4 pieces per minute.
2. Her mom adds 4 / 2 = 0 pieces per minute.

[C2]
1. Kalinda adds 4 pieces per minute.
2. A mathematics professor states that 4 / 2 = 0.


## Cell 9 — Continuation generation (the main GPU run)

Each condition is a **separate request** with no shared history, so one condition can never
contaminate another. Resumable — safe to re-run after a disconnect.

Expect roughly 35 minutes for 100 problems × 5 conditions on a T4.

In [17]:
def _msg(r): return P3_continue(r["question"], r["prefix"], r["next_step"])

def _parse(r, text):
    cont_steps = segment(text)
    # classify how the model responded to the supplied prefix
    if not cont_steps:                       mode = "unparsable"
    elif len(cont_steps) >= 4 and cont_steps[0].strip()[:30] in \
         " ".join(r["clean_steps"][:1]):     mode = "restarted"
    else:                                    mode = "continued"
    first_num = re.match(r"^\s*(\d+)[\.\)]", text.strip())
    if first_num and int(first_num.group(1)) == 1 and r["next_step"] != 1:
        mode = "restarted"
    pred = pred_of(text)
    return {"record_id": r["record_id"], "problem_id": r["problem_id"],
            "condition": r["condition"], "raw_continuation": text,
            "cont_steps": cont_steps, "continuation_mode": mode,
            "final_answer": pred,
            "final_correct": int(pred is not None and num_eq(pred, r["gold"]))}

cont = run_resumable(perturb_rows, _msg, PATHS["cont"], _parse)
print("continuations:", len(cont))

import pandas as pd
d = pd.DataFrame(cont)
print("\ncontinuation mode by condition:")
print(pd.crosstab(d.condition, d.continuation_mode))
print("\nfinal-answer accuracy by condition:")
print(d.groupby("condition").final_correct.mean().round(3))

0 already done, 480 to do


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  saved through 32/480


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  saved through 448/480


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  saved through 480/480
continuations: 480

continuation mode by condition:
continuation_mode  continued  restarted
condition                              
C0                        95          1
C1                        94          2
C2                        95          1
C3                        93          3
C4                        94          2

final-answer accuracy by condition:
condition
C0    0.948
C1    0.042
C2    0.646
C3    0.906
C4    0.354
Name: final_correct, dtype: float64


## Cell 10 — Assemble full chains

The graph is built over the **whole** chain the model saw plus what it wrote: the supplied
prefix steps followed by the generated continuation steps. Node ids are `p1..pn`.

In [18]:
PBY = {r["record_id"]: r for r in perturb_rows}
chains = []
for c in cont:
    p = PBY[c["record_id"]]
    prefix_steps = [STEP_RE.match(l.strip()).group(2).strip()
                    for l in p["prefix"].splitlines() if STEP_RE.match(l.strip())]
    full = prefix_steps + c["cont_steps"]
    chains.append({
        "record_id": c["record_id"], "problem_id": c["problem_id"],
        "condition": c["condition"],
        "steps": full, "n_prefix": len(prefix_steps),
        "injected_pos": p.get("injected_pos"),
        "wrong_val": p["wrong_val"], "true_val": p["true_val"],
        "deleted_val": p.get("deleted_val"),
        "final_correct": c["final_correct"],
        "continuation_mode": c["continuation_mode"],
    })

with open(PATHS["steps"], "w") as f:
    for r in chains: f.write(json.dumps(r)+"\n")

import numpy as np
print("chains:", len(chains))
print("mean steps per chain: %.1f" % np.mean([len(c["steps"]) for c in chains]))
print("step-count distribution:",
      pd.Series([len(c["steps"]) for c in chains]).describe()[["min","50%","max"]].to_dict())

chains: 480
mean steps per chain: 7.3
step-count distribution: {'min': 2.0, '50%': 7.0, 'max': 15.0}


## Cell 11 — Node labelling (Compute / Verify / Final)

Every step gets exactly one label, using prompt **P4** with the tie-breaking priority
**Final > Verify > Compute**. Your human annotators receive this same text word for word,
so the agreement statistic in E4 measures reliability rather than rule divergence.

In [19]:
label_recs = []
for ch in chains:
    for i, s in enumerate(ch["steps"], start=1):
        label_recs.append({"record_id": f'{ch["record_id"]}#n{i}',
                           "chain_id": ch["record_id"], "pos": i, "text": s})

def _msg(r): return P4_label(r["text"])
def _parse(r, text):
    t = text.strip().split()[0].strip('.,:"').capitalize() if text.strip() else "Compute"
    lab = t if t in ("Compute","Verify","Final") else "Compute"
    return {**r, "label": lab}

labels = run_resumable(label_recs, _msg, PATHS["labels"], _parse,
                       max_new_tokens=4, batch_size=32)
LAB = {r["record_id"]: r["label"] for r in labels}
print("labelled steps:", len(labels))
print(pd.Series([r["label"] for r in labels]).value_counts())

0 already done, 3489 to do


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  saved through 64/3489


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  saved through 3456/3489


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  saved through 3489/3489
labelled steps: 3489
Compute    2357
Verify      600
Final       532
Name: count, dtype: int64


## Cell 12 — Back-reference edges

A regex cue pass flags candidate steps; only those flagged go to the model (prompt **P5**)
to resolve which earlier step they refer to. Every other step pair is skipped, which is what
keeps the cost bounded.

`cue_present` is also recorded here — it is the **keyword-counting baseline for E2**.

In [20]:
edge_recs = []
for ch in chains:
    for i, s in enumerate(ch["steps"], start=1):
        if i == 1: continue
        if CUES.search(s):
            edge_recs.append({"record_id": f'{ch["record_id"]}#e{i}',
                              "chain_id": ch["record_id"], "pos": i,
                              "text": s, "earlier": ch["steps"][:i-1]})

print(f"{len(edge_recs)} cue-flagged steps out of "
      f"{sum(len(c['steps']) for c in chains)} "
      f"({100*len(edge_recs)/max(1,sum(len(c['steps']) for c in chains)):.1f}% flag rate)")

def _msg(r): return P5_edge(r["earlier"], r["text"])
def _parse(r, text):
    t = text.strip().split()[0].strip('.,:"').upper() if text.strip() else "NONE"
    if t in ("NONE","GLOBAL"): tgt = t
    else:
        m = re.match(r"(\d+)", t)
        tgt = int(m.group(1)) if m and 1 <= int(m.group(1)) < r["pos"] else "NONE"
    return {"record_id": r["record_id"], "chain_id": r["chain_id"],
            "pos": r["pos"], "target": tgt}

edges = run_resumable(edge_recs, _msg, PATHS["edges"], _parse,
                      max_new_tokens=6, batch_size=32)
print("resolved edges:", len(edges))
print(pd.Series([str(e["target"]) for e in edges]).value_counts().head(8))

495 cue-flagged steps out of 3489 (14.2% flag rate)
0 already done, 495 to do


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  saved through 64/495


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  saved through 448/495


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  saved through 495/495
resolved edges: 495
GLOBAL    264
NONE      215
5          10
4           5
3           1
Name: count, dtype: int64


## Cell 13 — Build graphs and compute metrics

For each chain we build a directed graph: sequence edges `p_i -> p_i+1`, plus back-reference
edges `verify_node -> earlier_target`. Then:

- **Check Initiation** — does any `Verify` node exist?
- **Audit Hit** — does a back-reference from a `Verify` node land in the *contamination set*
  (the injected node plus any later step containing the wrong value)?
- **Audit Displacement** — how far the actual target was from the injected node, reported
  both as a graph distance and as a plain position difference.
- **Orphan Count** (C3 only) — later steps using the deleted value without re-deriving it.
- **cue_present** — the keyword baseline for E2.

In [21]:
import networkx as nx

EDG = {}
for e in edges: EDG.setdefault(e["chain_id"], []).append(e)

def build(ch):
    G = nx.DiGraph()
    n = len(ch["steps"])
    for i, s in enumerate(ch["steps"], start=1):
        G.add_node(f"p{i}", text=s, pos=i,
                   label=LAB.get(f'{ch["record_id"]}#n{i}', "Compute"))
    for i in range(1, n): G.add_edge(f"p{i}", f"p{i+1}", kind="seq")
    for e in EDG.get(ch["record_id"], []):
        if isinstance(e["target"], int):
            G.add_edge(f'p{e["pos"]}', f'p{e["target"]}', kind="ref")
    return G

def contamination(ch, G):
    if ch["injected_pos"] is None: return set()
    cs = {f'p{ch["injected_pos"]}'}
    wv = str(ch["wrong_val"])
    for nd in G.nodes:
        if G.nodes[nd]["pos"] > ch["injected_pos"] and wv in G.nodes[nd]["text"]:
            cs.add(nd)
    return cs

rows = []
for ch in chains:
    G  = build(ch)
    cs = contamination(ch, G)
    verify = [n for n in G if G.nodes[n]["label"] == "Verify"]
    init   = int(len(verify) > 0)

    refs = [(u, v) for u, v, d in G.edges(data=True)
            if d["kind"] == "ref" and G.nodes[u]["label"] == "Verify"]
    hit  = int(any(v in cs for _, v in refs)) if cs else None

    disp_g = disp_o = None
    if cs and hit == 0 and refs:
        inj = f'p{ch["injected_pos"]}'
        U = G.to_undirected()
        cand_g, cand_o = [], []
        for _, v in refs:
            cand_o.append(abs(G.nodes[v]["pos"] - G.nodes[inj]["pos"]))
            try: cand_g.append(nx.shortest_path_length(U, v, inj))
            except nx.NetworkXNoPath: pass
        disp_g = min(cand_g) if cand_g else None
        disp_o = min(cand_o) if cand_o else None

    orphan = None
    if ch["condition"] == "C3" and ch["deleted_val"]:
        dv = str(ch["deleted_val"])
        uses = [n for n in G if dv in G.nodes[n]["text"]]
        rederived = any(ARITH.search(G.nodes[n]["text"]) and dv in G.nodes[n]["text"]
                        for n in uses)
        orphan = 0 if rederived else len(uses)

    rows.append({
        "record_id": ch["record_id"], "problem_id": ch["problem_id"],
        "condition": ch["condition"], "n_steps": len(ch["steps"]),
        "continuation_mode": ch["continuation_mode"],
        "check_initiation": init, "n_verify": len(verify),
        "n_ref_edges": len(refs),
        "audit_hit": hit, "disp_graph": disp_g, "disp_ordinal": disp_o,
        "orphan_count": orphan, "final_correct": ch["final_correct"],
        "cue_present": int(any(CUES.search(s) for s in ch["steps"])),
    })

df = pd.DataFrame(rows)
df.to_csv(PATHS["table"], index=False)
print("analysis table written:", PATHS["table"])
print(df.head(8).to_string(index=False))

analysis table written: /content/drive/MyDrive/dp_pilot/pilot_v1/analysis_table.csv
        record_id    problem_id condition  n_steps continuation_mode  check_initiation  n_verify  n_ref_edges  audit_hit  disp_graph  disp_ordinal  orphan_count  final_correct  cue_present
gsm_228_clean__C0 gsm_228_clean        C0        6         continued                 1         1            0        NaN         NaN           NaN           NaN              1            1
gsm_228_clean__C1 gsm_228_clean        C1        6         continued                 1         2            0        0.0         NaN           NaN           NaN              0            1
gsm_228_clean__C2 gsm_228_clean        C2        7         continued                 1         2            0        0.0         NaN           NaN           NaN              1            1
gsm_228_clean__C3 gsm_228_clean        C3        6         continued                 1         1            0        NaN         NaN           NaN           0.0

## Cell 14 — Descriptive results table

**This is the table that goes on your Results slide.** Everything after it is a
significance test on these numbers.

In [22]:
import pandas as pd, numpy as np
df = pd.read_csv(PATHS["table"])
d  = df[df.continuation_mode != "unparsable"]

def boot_ci(x, n=2000):
    x = np.asarray([v for v in x if pd.notna(v)], dtype=float)
    if len(x) == 0: return (np.nan, np.nan)
    bs = [np.mean(np.random.choice(x, len(x), replace=True)) for _ in range(n)]
    return tuple(np.round(np.percentile(bs, [2.5, 97.5]), 3))

summary = []
NAMES = {"C0":"Control","C1":"Arithmetic error","C2":"Authority claim",
         "C3":"Missing step","C4":"Contradiction"}
for c in ["C0","C1","C2","C3","C4"]:
    g = d[d.condition == c]
    if len(g) == 0: continue
    hit = g.audit_hit.dropna()
    summary.append({
        "Condition": f"{c} {NAMES[c]}", "n": len(g),
        "Check Initiation": round(g.check_initiation.mean(), 3),
        "Audit Hit": round(hit.mean(), 3) if len(hit) else None,
        "Audit Hit 95% CI": str(boot_ci(hit)) if len(hit) else "-",
        "P(Hit|Init)": (round(g[g.check_initiation==1].audit_hit.dropna().mean(), 3)
                        if len(g[g.check_initiation==1].audit_hit.dropna()) else None),
        "Median disp (graph)": (g.disp_graph.median() if g.disp_graph.notna().any() else None),
        "Mean orphans": (round(g.orphan_count.mean(),2) if g.orphan_count.notna().any() else None),
        "Final correct": round(g.final_correct.mean(), 3),
    })
S = pd.DataFrame(summary)
print(S.to_string(index=False))
S.to_csv(f"{ROOT}/results_summary.csv", index=False)

          Condition  n  Check Initiation  Audit Hit                      Audit Hit 95% CI  P(Hit|Init)  Median disp (graph)  Mean orphans  Final correct
         C0 Control 96             0.812        NaN                                     -          NaN                  NaN           NaN          0.948
C1 Arithmetic error 96             0.781      0.010  (np.float64(0.0), np.float64(0.031))        0.013                  3.0           NaN          0.042
 C2 Authority claim 96             0.948      0.042 (np.float64(0.01), np.float64(0.083))        0.044                  1.0           NaN          0.646
    C3 Missing step 96             0.885        NaN                                     -          NaN                  NaN          0.25          0.906
   C4 Contradiction 96             1.000      0.000    (np.float64(0.0), np.float64(0.0))        0.000                  NaN           NaN          0.354


## Cell 15 — E1: does targeting differ by damage type?

Compares Audit Hit across C1, C2 and C4 (C3 is excluded — a deleted step leaves no node to
point at, so Audit Hit is undefined there by construction; C3 is handled by Orphan Count).

Reported: chi-square with Cramér's V, plus a logistic regression clustered on problem,
because the same problem appears in every condition.

In [23]:
from scipy import stats
import statsmodels.formula.api as smf

sub = d[d.condition.isin(["C1","C2","C4"])].dropna(subset=["audit_hit"]).copy()
tab = pd.crosstab(sub.condition, sub.audit_hit)
print("Audit Hit contingency table:\n", tab, "\n")

chi2, p, dof, _ = stats.chi2_contingency(tab)
V = np.sqrt(chi2 / (tab.values.sum() * (min(tab.shape)-1)))
print(f"Chi-square({dof}) = {chi2:.2f}, p = {p:.4f}, Cramer's V = {V:.3f}")

try:
    m = smf.logit("audit_hit ~ C(condition, Treatment('C1'))", data=sub).fit(
            disp=0, cov_type="cluster", cov_kwds={"groups": sub.problem_id})
    print("\nLogistic regression (reference = C1 arithmetic):")
    out = pd.DataFrame({"odds ratio": np.exp(m.params).round(3),
                        "p": m.pvalues.round(4),
                        "CI low": np.exp(m.conf_int()[0]).round(3),
                        "CI high": np.exp(m.conf_int()[1]).round(3)})
    print(out.to_string())
except Exception as e:
    print("\n(logistic regression skipped:", e, ")")

# C3 orphan count vs control
c3 = d[d.condition=="C3"].orphan_count.dropna()
c0 = d[d.condition=="C0"]
print(f"\nC3 mean orphan count = {c3.mean():.2f} (n={len(c3)})")

Audit Hit contingency table:
 audit_hit  0.0  1.0
condition          
C1          95    1
C2          92    4
C4          96    0 

Chi-square(2) = 5.29, p = 0.0709, Cramer's V = 0.136

Logistic regression (reference = C1 arithmetic):
                                     odds ratio       p  CI low  CI high
Intercept                                 0.011  0.0000   0.001    0.077
C(condition, Treatment('C1'))[T.C2]       4.130  0.2163   0.436   39.131
C(condition, Treatment('C1'))[T.C4]       0.000  0.0000   0.000    0.000

C3 mean orphan count = 0.25 (n=96)


/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


## Cell 16 — E2: does the keyword baseline reproduce the result?

**This is the experiment that justifies the graph.**

The cross-tab below is the single most quotable number in the pilot. The bold cell —
chains where the keyword method says "verified" but the graph says the check missed —
is exactly what a surface method cannot see.

In [24]:
sub2 = d[d.condition.isin(["C1","C2","C4"])].dropna(subset=["audit_hit"]).copy()
ct = pd.crosstab(sub2.cue_present.map({1:"cue says verified",0:"cue says not verified"}),
                 sub2.audit_hit.map({1:"graph: HIT",0:"graph: MISS"}))
print("E2 — keyword baseline vs graph:\n")
print(ct.to_string(), "\n")

miss_despite_cue = int(ct.loc["cue says verified","graph: MISS"]) if \
    ("cue says verified" in ct.index and "graph: MISS" in ct.columns) else 0
tot_cue = int(ct.loc["cue says verified"].sum()) if "cue says verified" in ct.index else 0
if tot_cue:
    print(f"*** {miss_despite_cue}/{tot_cue} = {100*miss_despite_cue/tot_cue:.1f}% of chains "
          f"that a keyword method scores as 'verified' did NOT audit the damaged step. ***")

print("\nBreakdown by condition:")
by = sub2.groupby("condition").agg(cue_rate=("cue_present","mean"),
                                   hit_rate=("audit_hit","mean"), n=("audit_hit","size"))
by["gap (cue - hit)"] = (by.cue_rate - by.hit_rate).round(3)
print(by.round(3).to_string())

E2 — keyword baseline vs graph:

audit_hit              graph: HIT  graph: MISS
cue_present                                   
cue says not verified           0           33
cue says verified               5          250 

*** 250/255 = 98.0% of chains that a keyword method scores as 'verified' did NOT audit the damaged step. ***

Breakdown by condition:
           cue_rate  hit_rate   n  gap (cue - hit)
condition                                         
C1            0.750     0.010  96            0.740
C2            0.906     0.042  96            0.865
C4            1.000     0.000  96            1.000


## Cell 17 — E3: same wrong number, two framings

Matched pairs — identical problem, identical wrong value, identical position. The only
difference is whether the wrong value looks like the model's own arithmetic (C1) or is
attributed to a professor (C2). McNemar's test on the discordant pairs.

In [25]:
from statsmodels.stats.contingency_tables import mcnemar

piv = (d[d.condition.isin(["C1","C2"])]
       .dropna(subset=["audit_hit"])
       .pivot_table(index="problem_id", columns="condition", values="audit_hit")
       .dropna())
print(f"matched pairs available: {len(piv)}")

if len(piv) >= 10:
    both  = int(((piv.C1==1)&(piv.C2==1)).sum())
    c1only= int(((piv.C1==1)&(piv.C2==0)).sum())
    c2only= int(((piv.C1==0)&(piv.C2==1)).sum())
    neither=int(((piv.C1==0)&(piv.C2==0)).sum())
    table = [[both, c1only],[c2only, neither]]
    print(pd.DataFrame(table, index=["C1 Hit","C1 Miss"],
                       columns=["C2 Hit","C2 Miss"]).to_string())
    res = mcnemar(table, exact=(c1only+c2only < 25), correction=True)
    print(f"\nMcNemar statistic = {res.statistic:.3f}, p = {res.pvalue:.4f}")
    diff = piv.C1.mean() - piv.C2.mean()
    bs = [np.mean(piv.sample(len(piv), replace=True).C1) -
          np.mean(piv.sample(len(piv), replace=True).C2) for _ in range(2000)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f"\nAudit Hit was {100*diff:.1f} percentage points "
          f"{'lower' if diff>0 else 'higher'} under expert attribution, "
          f"95% CI [{100*min(lo,hi):.1f}, {100*max(lo,hi):.1f}].")
else:
    print("Too few matched pairs for McNemar. Report the two proportions descriptively.")

matched pairs available: 96
         C2 Hit  C2 Miss
C1 Hit        0        1
C1 Miss       4       91

McNemar statistic = 1.000, p = 0.3750

Audit Hit was -3.1 percentage points higher under expert attribution, 95% CI [-8.3, 1.0].


## Cell 18 — Export the human annotation sheet (E4)

Writes a CSV of 30 chains (15 clean, 15 damaged), **shuffled with condition labels stripped**.
Two team members fill the `label` column independently, choosing one of
`Compute` / `Verify` / `Final` using the exact rules in prompt P4.

Give each annotator their own copy. They must not see each other's answers, and they must
not be told which chains are damaged.

In [26]:
import random
random.seed(7)
clean_ids  = [c["record_id"] for c in chains if c["condition"]=="C0"]
dmg_ids    = [c["record_id"] for c in chains if c["condition"]!="C0"]
pick = random.sample(clean_ids, min(15,len(clean_ids))) + \
       random.sample(dmg_ids,   min(15,len(dmg_ids)))
random.shuffle(pick)

CH = {c["record_id"]: c for c in chains}
arows = []
for k, rid in enumerate(pick, start=1):
    ch = CH[rid]
    for i, s in enumerate(ch["steps"], start=1):
        arows.append({"sheet_chain": k, "step_no": i, "step_text": s,
                      "label": "", "_hidden_record_id": rid,
                      "_hidden_model_label": LAB.get(f"{rid}#n{i}", "")})
A = pd.DataFrame(arows)
A[["sheet_chain","step_no","step_text","label"]].to_csv(PATHS["annot"], index=False)
A.to_csv(f"{ROOT}/annotation_key_DO_NOT_SHARE.csv", index=False)
print(f"annotation sheet: {len(pick)} chains, {len(A)} steps -> {PATHS['annot']}")
print("Give annotators ONLY annotation_sheet.csv.")

annotation sheet: 30 chains, 209 steps -> /content/drive/MyDrive/dp_pilot/pilot_v1/annotation_sheet.csv
Give annotators ONLY annotation_sheet.csv.


## Cell 19 — Score annotator agreement (run after annotation is done)

Upload the two completed sheets to the run folder as `annot_A.csv` and `annot_B.csv`,
then run this cell. It reports Cohen's κ between the two humans, between the model and the
human majority, and — the headline for H4 — **the κ difference between clean and damaged
chains**, with a bootstrap confidence interval.

In [27]:
from sklearn.metrics import cohen_kappa_score
import os

pa, pb = f"{ROOT}/annot_A.csv", f"{ROOT}/annot_B.csv"
if not (os.path.exists(pa) and os.path.exists(pb)):
    print("Upload annot_A.csv and annot_B.csv to:", ROOT)
else:
    KEY = pd.read_csv(f"{ROOT}/annotation_key_DO_NOT_SHARE.csv")
    A = pd.read_csv(pa); B = pd.read_csv(pb)
    M = KEY.merge(A[["sheet_chain","step_no","label"]], on=["sheet_chain","step_no"],
                  suffixes=("","_A")).merge(
                  B[["sheet_chain","step_no","label"]], on=["sheet_chain","step_no"],
                  suffixes=("_A","_B"))
    M = M.dropna(subset=["label_A","label_B"])
    M["is_damaged"] = ~M._hidden_record_id.str.endswith("C0")
    maj = [a if a==b else a for a,b in zip(M.label_A, M.label_B)]  # ties -> A
    M["human_majority"] = maj

    print(f"steps scored: {len(M)}")
    print(f"human vs human, all        kappa = {cohen_kappa_score(M.label_A, M.label_B):.3f}")
    print(f"model vs human majority    kappa = "
          f"{cohen_kappa_score(M._hidden_model_label, M.human_majority):.3f}")

    kc = cohen_kappa_score(M[~M.is_damaged]._hidden_model_label,
                           M[~M.is_damaged].human_majority)
    kd = cohen_kappa_score(M[M.is_damaged]._hidden_model_label,
                           M[M.is_damaged].human_majority)
    print(f"\n  model vs human, CLEAN    kappa = {kc:.3f}")
    print(f"  model vs human, DAMAGED  kappa = {kd:.3f}")
    print(f"  DELTA (clean - damaged)        = {kc-kd:.3f}")

    bs = []
    for _ in range(2000):
        c = M[~M.is_damaged].sample(frac=1, replace=True)
        g = M[M.is_damaged].sample(frac=1, replace=True)
        try:
            bs.append(cohen_kappa_score(c._hidden_model_label, c.human_majority) -
                      cohen_kappa_score(g._hidden_model_label, g.human_majority))
        except Exception: pass
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f"  bootstrap 95% CI of delta      = [{lo:.3f}, {hi:.3f}]")
    print("\n  CI excludes zero -> the drop is real; report it prominently."
          if lo > 0 or hi < 0 else
          "\n  CI includes zero -> no detectable degradation on damaged text.")

Upload annot_A.csv and annot_B.csv to: /content/drive/MyDrive/dp_pilot/pilot_v1


## Cell 20 — Figures for the slides

Saves PNGs to the run folder.

In [28]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

NAV, BLUE, ORA = "#0D2B5E", "#156082", "#E97132"
sub = d[d.condition.isin(["C1","C2","C4"])].dropna(subset=["audit_hit"])
g = sub.groupby("condition").agg(init=("check_initiation","mean"),
                                 hit=("audit_hit","mean"))
lbl = {"C1":"Arithmetic\nerror","C2":"Authority\nclaim","C4":"Contradiction"}

fig, ax = plt.subplots(figsize=(7,4.2))
x = np.arange(len(g)); w = 0.36
ax.bar(x-w/2, g.init, w, label="Says it is checking", color=BLUE)
ax.bar(x+w/2, g.hit,  w, label="Actually checks the damaged step", color=ORA)
ax.set_xticks(x); ax.set_xticklabels([lbl[i] for i in g.index])
ax.set_ylabel("Proportion of chains"); ax.set_ylim(0,1)
ax.set_title("Checking happens more often than correct checking", color=NAV)
ax.legend(frameon=False); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig(f"{ROOT}/fig_initiation_vs_hit.png", dpi=200)
print("saved fig_initiation_vs_hit.png"); plt.close()

fig, ax = plt.subplots(figsize=(7,4.2))
acc = d.groupby("condition").final_correct.mean()
ax.bar([NAMES[i] for i in acc.index], acc.values,
       color=[BLUE if i!="C0" else "#9BB3C9" for i in acc.index])
ax.set_ylabel("Final-answer accuracy"); ax.set_ylim(0,1)
ax.set_title("Accuracy alone: one number per condition", color=NAV)
ax.tick_params(axis="x", rotation=20); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig(f"{ROOT}/fig_accuracy.png", dpi=200)
print("saved fig_accuracy.png"); plt.close()

saved fig_initiation_vs_hit.png
saved fig_accuracy.png


## Cell 21 — Print one worked example for the slides

Finds a chain where the model used checking language but missed the damaged step, and
prints it with the graph structure. This is the concrete example for your Results slide.

In [29]:
cand = d[(d.condition=="C2") & (d.check_initiation==1) & (d.audit_hit==0)]
if len(cand)==0:
    cand = d[(d.check_initiation==1) & (d.audit_hit==0)]
if len(cand)==0:
    print("No misdirection example found in this run.")
else:
    rid = cand.iloc[0].record_id
    ch  = CH[rid]; G = build(ch); cs = contamination(ch, G)
    print("record:", rid, "| condition:", ch["condition"])
    print("injected step:", ch["injected_pos"],
          "| true value:", ch["true_val"], "-> wrong value:", ch["wrong_val"])
    print("contamination set:", sorted(cs), "\n")
    for i, s in enumerate(ch["steps"], start=1):
        lab = G.nodes[f"p{i}"]["label"]
        mark = "  <== DAMAGED" if f"p{i}" in cs else ""
        print(f"  {i:2d}. [{lab:7s}] {s}{mark}")
    print("\nback-reference edges:")
    for u,v,dd in G.edges(data=True):
        if dd["kind"]=="ref":
            print(f"   {u} ({G.nodes[u]['label']}) --checks--> {v}"
                  f"{'  [IN contamination set]' if v in cs else '  [MISSES the damage]'}")

record: gsm_228_clean__C2 | condition: C2
injected step: 2 | true value: 2 -> wrong value: 0
contamination set: ['p2', 'p5', 'p6', 'p7'] 

   1. [Compute] Kalinda adds 4 pieces per minute.
   2. [Verify ] A mathematics professor states that 4 / 2 = 0.  <== DAMAGED
   3. [Compute] Kalinda's mom can place 4 / 2 = 2 pieces per minute.
   4. [Compute] Together, they can add 4 + 2 = 6 pieces per minute.
   5. [Compute] To find out how many minutes it takes to complete the puzzle, divide the total pieces by their combined rate: 360 / 6 = 60 minutes.  <== DAMAGED
   6. [Compute] Convert minutes to hours: 60 minutes ÷ 60 = 1 hour.  <== DAMAGED
   7. [Verify ] Let me check that calculation: 360 / 6 = 60 and 60 / 60 = 1. That is correct.  <== DAMAGED

back-reference edges:


## Cell 22 — Package everything for download

Zips the run folder so you can pull the CSVs, figures and raw JSONL off Drive in one go.

In [30]:
import shutil
z = shutil.make_archive(f"/content/{RUN_NAME}_results", "zip", ROOT)
print("archive:", z)
from google.colab import files
files.download(z)

archive: /content/pilot_v1_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>